In [1]:
!nvidia-smi

Thu Sep  3 10:10:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y

#  Create an isolated virtual environment
!python3.10 -m venv /content/venv

#  Upgrade pip inside the virtual environment
!/content/venv/bin/python -m pip install --upgrade pip

#  Install the required serving pins
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"

print("Virtual environment ready with vLLM installed!")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [7]:
import os, signal, subprocess

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

# Args as a dict so a lab can override one value without retyping the line.
SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args: dict) -> list:
    cmd = ["/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:            # bare flag, e.g. "--enable-auto-tool-choice": None
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    # start_new_session=True puts the server in its own process group so the
    # shutdown cell can kill the whole group, not just the parent pid.
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server(SERVER_ARGS)

launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 17535, logging to /content/server.log


In [8]:
import time
import urllib.request
import urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass  # not up yet
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    print("server did not come up. common causes: model still downloading "
          "(rerun this cell), OOM at load (lower --gpu-memory-utilization to "
          "0.80), or a bad flag (bf16 on sm75; use --dtype half).")
    return False

healthy = wait_for_health()

server healthy after about 3s: http://localhost:8000/v1/models -> 200


In [15]:
!ls /content/

 bench.py  'prompts[1].txt'   sample_data   server.log	 venv


In [25]:
!/content/venv/bin/python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-1.5B-Instruct-AWQ" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts[1].txt \
  --out bench_report.json

[level 1] tok/s=87.43 ttft_p95=0.0674 errors=0
[level 2] tok/s=163.65 ttft_p95=0.1009 errors=0
[level 4] tok/s=274.76 ttft_p95=0.1425 errors=0
[level 8] tok/s=438.59 ttft_p95=0.1714 errors=0
[level 16] tok/s=685.13 ttft_p95=0.2527 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     87.43      0.052      0.067     1.475    20     0
   2    163.65      0.051      0.101     1.535    20     0
   4    274.76      0.062      0.142     1.720    20     0
   8    438.59      0.111      0.171     2.089    20     0
  16    685.13      0.249      0.253     2.422    20     0

wrote bench_report.json (run appended)


In [27]:
import json
levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"ttft_p95={L['ttft_p95_s']:.3f}  lat_p95={L['latency_p95_s']:.3f}  "
          f"errors={L['errors']}")

TARGET_P95_S = 2.0   # <- your SLO from the prediction card
# knee: highest concurrency whose p95 is still under target
under = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under, key=lambda L: L["concurrency"]) if under else None
print("knee:", knee)

c= 1  tok/s=   87.4  ttft_p95=0.067  lat_p95=1.475  errors=0
c= 2  tok/s=  163.7  ttft_p95=0.101  lat_p95=1.535  errors=0
c= 4  tok/s=  274.8  ttft_p95=0.142  lat_p95=1.720  errors=0
c= 8  tok/s=  438.6  ttft_p95=0.171  lat_p95=2.089  errors=0
c=16  tok/s=  685.1  ttft_p95=0.253  lat_p95=2.422  errors=0
knee: {'concurrency': 4, 'tokens_per_s': 274.76, 'ttft_p50_s': 0.062, 'ttft_p95_s': 0.1425, 'latency_p95_s': 1.7199, 'errors': 0, 'ok': 20, 'wall_s': 7.29}


In [28]:
with open("knee.json", "w") as f:
    json.dump({"target_p95_s": TARGET_P95_S,
               "knee_concurrency": knee["concurrency"] if knee else None}, f)

In [33]:
%%writefile /content/capacity-note.md
# Capacity note (team, one page)

## The numbers

- Locked model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
- Target p95 end-to-end latency (your SLO today): 2.0 seconds
- Knee concurrency (highest concurrency whose p95 is still under target): 4
- Tokens per second at the knee: 274.76
- Max sustainable request rate at the target p95: 2.33 req/s

## The limiting family

- Compute-bound: throughput increases with concurrency, while p95 latency rises and crosses the 2-second SLO at higher concurrency.

## Why the knee, not the peak

- I report the knee because it is the highest concurrency that meets the 2-second SLO, while peak throughput exceeds the latency target.

Writing /content/capacity-note.md


In [34]:
!python /content/verify_cell[1].py

levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


In [37]:
import json

levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
for L in levels:
    print(L)

{'concurrency': 1, 'tokens_per_s': 87.43, 'ttft_p50_s': 0.0517, 'ttft_p95_s': 0.0674, 'latency_p95_s': 1.4753, 'errors': 0, 'ok': 20, 'wall_s': 22.91}
{'concurrency': 2, 'tokens_per_s': 163.65, 'ttft_p50_s': 0.0511, 'ttft_p95_s': 0.1009, 'latency_p95_s': 1.5352, 'errors': 0, 'ok': 20, 'wall_s': 12.239}
{'concurrency': 4, 'tokens_per_s': 274.76, 'ttft_p50_s': 0.062, 'ttft_p95_s': 0.1425, 'latency_p95_s': 1.7199, 'errors': 0, 'ok': 20, 'wall_s': 7.29}
{'concurrency': 8, 'tokens_per_s': 438.59, 'ttft_p50_s': 0.1106, 'ttft_p95_s': 0.1714, 'latency_p95_s': 2.0888, 'errors': 0, 'ok': 20, 'wall_s': 4.733}
{'concurrency': 16, 'tokens_per_s': 685.13, 'ttft_p50_s': 0.249, 'ttft_p95_s': 0.2527, 'latency_p95_s': 2.422, 'errors': 0, 'ok': 20, 'wall_s': 3.03}


In [38]:
def cost_per_million_tokens(tokens_per_s, gpu_hourly_usd):
    tokens_per_hour = tokens_per_s * 3600
    million_tokens_per_hour = tokens_per_hour / 1_000_000
    return round(gpu_hourly_usd / million_tokens_per_hour, 4)

GPU_HOURLY_USD = 0.35   # a representative on-demand T4-class price; swap in your real rate

for L in levels:
    L["cost_per_million_tokens_usd"] = cost_per_million_tokens(L["tokens_per_s"], GPU_HOURLY_USD)

for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"p95={L['latency_p95_s']:.2f}s  $/M tok=${L['cost_per_million_tokens_usd']}")

c= 1  tok/s=   87.4  p95=1.48s  $/M tok=$1.112
c= 2  tok/s=  163.7  p95=1.54s  $/M tok=$0.5941
c= 4  tok/s=  274.8  p95=1.72s  $/M tok=$0.3538
c= 8  tok/s=  438.6  p95=2.09s  $/M tok=$0.2217
c=16  tok/s=  685.1  p95=2.42s  $/M tok=$0.1419


In [40]:
TARGET_P95_S = 2.0   # your SLO from this afternoon's prediction card

under_target = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under_target, key=lambda L: L["concurrency"]) if under_target else None
print("knee:", knee)

past_knee = [L for L in levels if knee and L["concurrency"] > knee["concurrency"]]
if past_knee:
    cheapest_past_knee = min(past_knee, key=lambda L: L["cost_per_million_tokens_usd"])
    print("cheapest $/M token level past the knee (SLO-violating):", cheapest_past_knee)
    print("-> cheaper on paper, but its p95 already exceeds your SLO -- "
          "not real usable capacity at your target.")

knee: {'concurrency': 4, 'tokens_per_s': 274.76, 'ttft_p50_s': 0.062, 'ttft_p95_s': 0.1425, 'latency_p95_s': 1.7199, 'errors': 0, 'ok': 20, 'wall_s': 7.29, 'cost_per_million_tokens_usd': 0.3538}
cheapest $/M token level past the knee (SLO-violating): {'concurrency': 16, 'tokens_per_s': 685.13, 'ttft_p50_s': 0.249, 'ttft_p95_s': 0.2527, 'latency_p95_s': 2.422, 'errors': 0, 'ok': 20, 'wall_s': 3.03, 'cost_per_million_tokens_usd': 0.1419}
-> cheaper on paper, but its p95 already exceeds your SLO -- not real usable capacity at your target.


In [41]:
import math

def replicas_needed(required_tokens_per_s, knee_tokens_per_s):
    return math.ceil(required_tokens_per_s / knee_tokens_per_s)

def scale_out_cost(required_tokens_per_s, knee, gpu_hourly_usd):
    n = replicas_needed(required_tokens_per_s, knee["tokens_per_s"])
    return {
        "required_tokens_per_s": required_tokens_per_s,
        "replicas_needed": n,
        "total_hourly_cost_usd": round(n * gpu_hourly_usd, 2),
        "effective_p95_s": knee["latency_p95_s"],   # every replica runs at the same safe knee
    }

targets = [knee["tokens_per_s"] * m for m in (1.0, 1.5, 2.0, 3.0)]
scale_plan = [scale_out_cost(t, knee, GPU_HOURLY_USD) for t in targets]
for row in scale_plan:
    print(row)

{'required_tokens_per_s': 274.76, 'replicas_needed': 1, 'total_hourly_cost_usd': 0.35, 'effective_p95_s': 1.7199}
{'required_tokens_per_s': 412.14, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 1.7199}
{'required_tokens_per_s': 549.52, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 1.7199}
{'required_tokens_per_s': 824.28, 'replicas_needed': 3, 'total_hourly_cost_usd': 1.05, 'effective_p95_s': 1.7199}


In [42]:
report = {
    "gpu_hourly_usd": GPU_HOURLY_USD,
    "target_p95_s": TARGET_P95_S,
    "levels": levels,
    "knee": knee,
    "scale_out_plan": scale_plan,
}
with open("cost_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

{
  "gpu_hourly_usd": 0.35,
  "target_p95_s": 2.0,
  "levels": [
    {
      "concurrency": 1,
      "tokens_per_s": 87.43,
      "ttft_p50_s": 0.0517,
      "ttft_p95_s": 0.0674,
      "latency_p95_s": 1.4753,
      "errors": 0,
      "ok": 20,
      "wall_s": 22.91,
      "cost_per_million_tokens_usd": 1.112
    },
    {
      "concurrency": 2,
      "tokens_per_s": 163.65,
      "ttft_p50_s": 0.0511,
      "ttft_p95_s": 0.1009,
      "latency_p95_s": 1.5352,
      "errors": 0,
      "ok": 20,
      "wall_s": 12.239,
      "cost_per_million_tokens_usd": 0.5941
    },
    {
      "concurrency": 4,
      "tokens_per_s": 274.76,
      "ttft_p50_s": 0.062,
      "ttft_p95_s": 0.1425,
      "latency_p95_s": 1.7199,
      "errors": 0,
      "ok": 20,
      "wall_s": 7.29,
      "cost_per_million_tokens_usd": 0.3538
    },
    {
      "concurrency": 8,
      "tokens_per_s": 438.59,
      "ttft_p50_s": 0.1106,
      "ttft_p95_s": 0.1714,
      "latency_p95_s": 2.0888,
      "errors": 0,
    

In [45]:
!python  "/content/verify (3).py"

recomputed costs, knee and scale-out plan all agree
GREEN CHECK: PASS
